# Bayesian Strategy Evaluation

Every strategy evaluation in our research (notebooks 00-04) relies on **point estimates**:
a single Sharpe, a single CAGR, a single hit rate. This notebook adds the missing piece:
**uncertainty quantification**.

Using the toolkit from *Bayesian Statistics the Fun Way* (Will Kurt), we answer:

1. **How confident are we?** — Credible intervals for Sharpe ratios
2. **Is strategy A actually better than B?** — P(A beats B) from posterior sampling
3. **What's the true hit rate?** — Beta-Binomial posterior with uncertainty bands
4. **Is there real edge?** — Bayes factors for positive Sharpe
5. **Is our best parameter overfit?** — Bayesian model averaging across parameter grids
6. **Side-by-side comparison** — Augmented metrics table (frequentist + Bayesian)
7. **Regime probabilities** — Beta updating for smooth regime detection

In [ ]:
from _setup import *
from scipy import stats as sp_stats

from common.bayesian import (
    posterior_sharpe,
    sharpe_credible_interval,
    p_a_beats_b,
    beta_hit_rate,
    bayes_factor_positive_sharpe,
    parameter_robustness,
    compute_bayesian_metrics,
    format_bayesian_table,
    comparison_table,
)

## 1. Data & Baseline Strategies

Load daily bars and construct three baseline strategies for BTC-USD:
- **Buy & Hold** — the benchmark
- **SMA 5/40 Crossover** — fast/slow MA, long when fast > slow
- **Turtle System 1** — 20-day breakout entry, 10-day breakout exit

In [ ]:
SYMBOL = "BTC-USD"
START, END = "2017-01-01", "2026-12-31"

panel = load_daily_bars(start=START, end=END)
btc = panel.loc[panel["symbol"] == SYMBOL].copy().sort_values("ts").set_index("ts")
btc["ret"] = btc["close"].pct_change()
btc = btc.dropna(subset=["ret"])
print(f"BTC: {len(btc)} days, {btc.index[0].date()} to {btc.index[-1].date()}")

In [ ]:
# Buy & Hold: weight = 1.0 every day
bh_weights = pd.DataFrame(1.0, index=btc.index, columns=[SYMBOL])
bh_returns = pd.DataFrame(btc["ret"].values, index=btc.index, columns=[SYMBOL])

# SMA 5/40 crossover: long when fast > slow
fast_ma = btc["close"].rolling(5).mean()
slow_ma = btc["close"].rolling(40).mean()
ma_signal = (fast_ma > slow_ma).astype(float)
ma_weights = pd.DataFrame(ma_signal.values, index=btc.index, columns=[SYMBOL])

# Turtle System 1: 20-day high breakout entry, 10-day low breakout exit
entry_high = btc["high"].rolling(20).max().shift(1)
exit_low = btc["low"].rolling(10).min().shift(1)
turtle_pos = pd.Series(0.0, index=btc.index)
for i in range(1, len(btc)):
    if btc["close"].iloc[i] > entry_high.iloc[i] if pd.notna(entry_high.iloc[i]) else False:
        turtle_pos.iloc[i] = 1.0
    elif btc["close"].iloc[i] < exit_low.iloc[i] if pd.notna(exit_low.iloc[i]) else False:
        turtle_pos.iloc[i] = 0.0
    else:
        turtle_pos.iloc[i] = turtle_pos.iloc[i - 1]
turtle_weights = pd.DataFrame(turtle_pos.values, index=btc.index, columns=[SYMBOL])

# Run backtests
results = [
    quick_backtest(bh_weights, bh_returns, cost_bps=0, label="Buy & Hold"),
    quick_backtest(ma_weights, bh_returns, label="SMA 5/40"),
    quick_backtest(turtle_weights, bh_returns, label="Turtle Sys1"),
]
metrics_table(results)

## 2. Posterior Sharpe Distributions

Instead of a single Sharpe number, we draw 20,000 samples from the **posterior distribution**
of each strategy's Sharpe ratio. The prior is skeptical: mean=0, std=0.5 (most strategies
don't work). Short backtests get wide posteriors; long backtests converge to the point estimate.

This is the Normal-Inverse-Gamma conjugate model from Chapters 9-10 of the book.

In [ ]:
strategy_returns = {
    "Buy & Hold": results[0]["equity"].pct_change().dropna(),
    "SMA 5/40":   results[1]["equity"].pct_change().dropna(),
    "Turtle Sys1": results[2]["equity"].pct_change().dropna(),
}

fig, ax = plt.subplots(figsize=(12, 5))
colors_list = [NAVY, TEAL, RED]

for i, (name, ret) in enumerate(strategy_returns.items()):
    samples = posterior_sharpe(ret, n_samples=20_000)
    ax.hist(samples, bins=80, alpha=0.4, color=colors_list[i], label=name, density=True)
    ax.axvline(np.median(samples), color=colors_list[i], ls="--", lw=1.5)

ax.axvline(0, color="black", ls=":", lw=1, alpha=0.5)
ax.set_xlabel("Annualised Sharpe Ratio")
ax.set_ylabel("Posterior Density")
ax.set_title("Posterior Sharpe Distributions (skeptical prior: SR ~ N(0, 0.5))", fontweight="bold")
ax.legend(frameon=True, facecolor="white")
fig.tight_layout()
plt.show()

In [ ]:
print("Sharpe Credible Intervals (95%)")
print("=" * 65)
for name, ret in strategy_returns.items():
    ci = sharpe_credible_interval(ret)
    print(
        f"  {name:15s}  median={ci['median']:+.2f}  "
        f"CI=[{ci['lower']:+.2f}, {ci['upper']:+.2f}]  "
        f"P(SR>0)={ci['p_positive']:.1%}"
    )

## 3. Strategy Comparison: P(A beats B)

Instead of eyeballing Sharpe differences, we compute **P(Sharpe_A > Sharpe_B)** directly
from paired posterior draws. This handles the uncertainty that point estimates hide.

A probability near 50% means we genuinely can't tell which is better (honest answer).
A probability above 75% starts to be convincing.

In [ ]:
names = list(strategy_returns.keys())
ret_list = list(strategy_returns.values())

pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        result = p_a_beats_b(ret_list[i], ret_list[j])
        pairs.append((names[i], names[j], result))

print(comparison_table(pairs))

In [ ]:
# Visualise the Sharpe difference distribution for the most interesting pair
best_pair = max(pairs, key=lambda x: abs(x[2]["p_a_wins"] - 0.5))
name_a, name_b = best_pair[0], best_pair[1]

diff_samples = (
    posterior_sharpe(strategy_returns[name_a], n_samples=20_000)
    - posterior_sharpe(strategy_returns[name_b], n_samples=20_000)
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(diff_samples, bins=80, color=TEAL, alpha=0.6, density=True)
ax.axvline(0, color="black", ls=":", lw=1.5)
ax.axvline(np.median(diff_samples), color=RED, ls="--", lw=1.5, label=f"median={np.median(diff_samples):+.2f}")
p_a = float(np.mean(diff_samples > 0))
ax.set_title(f"Sharpe Difference: {name_a} minus {name_b}  |  P(A>B)={p_a:.1%}", fontweight="bold")
ax.set_xlabel("Sharpe(A) - Sharpe(B)")
ax.legend(frameon=True, facecolor="white")
fig.tight_layout()
plt.show()

## 4. Beta-Binomial Hit Rate

The hit rate (fraction of winning days) is a **proportion**, and the Beta distribution is
the natural Bayesian tool for proportions (Chapters 5-6 of the book).

- Prior: Beta(1, 1) = uniform ("I have no idea")
- Data: k wins out of n days
- Posterior: Beta(1 + k, 1 + n - k)

The posterior gives us credible intervals and P(hit rate > 50%).

In [ ]:
fig, axes = plt.subplots(1, len(strategy_returns), figsize=(14, 4), sharey=True)

for ax, (name, ret), color in zip(axes, strategy_returns.items(), [NAVY, TEAL, RED]):
    hr = beta_hit_rate(ret)
    
    # Plot posterior Beta distribution
    x = np.linspace(0.35, 0.65, 300)
    n_wins, n_total = hr["n_wins"], hr["n_total"]
    pdf = sp_stats.beta.pdf(x, 1 + n_wins, 1 + n_total - n_wins)
    ax.fill_between(x, pdf, alpha=0.3, color=color)
    ax.plot(x, pdf, color=color, lw=2)
    
    # Mark CI and 50% line
    ax.axvline(0.5, color="black", ls=":", alpha=0.5)
    ax.axvline(hr["posterior_mean"], color=color, ls="--", lw=1.5)
    ax.axvspan(hr["lower"], hr["upper"], alpha=0.1, color=color)
    
    ax.set_title(f"{name}\nmean={hr['posterior_mean']:.3f}, P(>50%)={hr['p_above_50']:.1%}", fontsize=10)
    ax.set_xlabel("Hit Rate")

axes[0].set_ylabel("Posterior Density")
fig.suptitle("Beta-Binomial Posterior for Hit Rate", fontweight="bold", fontsize=12)
fig.tight_layout()
plt.show()

## 5. Bayes Factors: Is There Real Edge?

A Bayes factor quantifies the **evidence** for H1 (Sharpe > 0) vs H0 (Sharpe <= 0).
Unlike p-values, Bayes factors can provide evidence *for* the null — telling us when
there's genuinely no edge, not just "insufficient evidence to reject."

Jeffreys' scale (from the book):
- BF < 1: evidence *against* positive edge
- 1-3: barely worth mentioning
- 3-10: moderate evidence
- 10-30: strong evidence
- 30-100: very strong evidence
- \>100: decisive

In [ ]:
print("Bayes Factors for Positive Sharpe")
print("=" * 65)
bf_results = []
for name, ret in strategy_returns.items():
    bf = bayes_factor_positive_sharpe(ret)
    bf_results.append((name, bf))
    print(
        f"  {name:15s}  BF={bf['bf']:>7.1f}  "
        f"P(SR>0)={bf['p_positive']:.1%}  "
        f"[{bf['interpretation']}]"
    )

In [ ]:
# Visualise Bayes factors
bf_names = [r[0] for r in bf_results]
bf_vals = [r[1]["bf"] for r in bf_results]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(bf_names, bf_vals, color=[NAVY, TEAL, RED])

# Reference lines for Jeffreys' thresholds
for threshold, label in [(3, "moderate"), (10, "strong"), (30, "very strong")]:
    ax.axvline(threshold, color=GRAY, ls="--", alpha=0.5)
    ax.text(threshold, len(bf_names) - 0.1, f" {label}", fontsize=8, color=GRAY, va="top")

ax.axvline(1, color="black", ls=":", lw=1)
ax.set_xlabel("Bayes Factor (H1: SR > 0)")
ax.set_title("Evidence Strength for Positive Edge", fontweight="bold")
fig.tight_layout()
plt.show()

## 6. Parameter Robustness: Bayesian Model Averaging

When we sweep parameters (e.g. MA lookback windows), picking the max-Sharpe parameter
is classic overfitting. **Bayesian model averaging** weights each parameter by its
approximate posterior probability, producing a robust expected Sharpe.

Key diagnostic: **effective_n_params** — how many parameter values carry meaningful
posterior weight. High = robust plateau. Low = fragile peak.

In [ ]:
# Run a parameter sweep: SMA fast/slow crossover on BTC
fast_periods = [3, 5, 8, 10, 15, 20]
slow_periods = [20, 30, 40, 50, 60, 80, 100]

sweep_metrics = {}
for fast in fast_periods:
    for slow in slow_periods:
        if fast >= slow:
            continue
        fast_sma = btc["close"].rolling(fast).mean()
        slow_sma = btc["close"].rolling(slow).mean()
        sig = (fast_sma > slow_sma).astype(float)
        w = pd.DataFrame(sig.values, index=btc.index, columns=[SYMBOL])
        res = quick_backtest(w, bh_returns, label=f"SMA {fast}/{slow}")
        if res["metrics"]:
            sweep_metrics[f"{fast}/{slow}"] = res["metrics"]

print(f"Swept {len(sweep_metrics)} parameter combinations")

In [ ]:
rob = parameter_robustness(sweep_metrics)

print("Bayesian Model Averaging — SMA Fast/Slow Sweep")
print("=" * 55)
print(f"  Best single param:       {rob['best_param']}  (Sharpe = {rob['best_sharpe']:.2f})")
print(f"  BMA-weighted Sharpe:     {rob['posterior_weighted_sharpe']:.2f}")
print(f"  Effective # params:      {rob['effective_n_params']:.1f} / {len(sweep_metrics)}")
print(f"  Concentration (HHI):     {rob['concentration']:.3f}")
print()
if rob['effective_n_params'] < 3:
    print("  >> WARNING: posterior concentrates on few params — result is fragile.")
else:
    print("  >> Posterior is spread across multiple params — reasonably robust.")

In [ ]:
# Plot posterior weights across parameters
pw = rob["param_weights"]
sorted_params = sorted(pw.keys(), key=lambda k: pw[k], reverse=True)
top_n = min(20, len(sorted_params))

fig, ax = plt.subplots(figsize=(12, 5))
labels = sorted_params[:top_n]
vals = [pw[k] for k in labels]
bar_colors = [RED if k == rob["best_param"] else TEAL for k in labels]
ax.bar(range(len(labels)), vals, color=bar_colors, alpha=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Posterior Weight")
ax.set_title(
    f"Parameter Posterior Weights (top {top_n})  |  "
    f"BMA Sharpe={rob['posterior_weighted_sharpe']:.2f}  |  "
    f"eff_params={rob['effective_n_params']:.1f}",
    fontweight="bold", fontsize=11,
)
fig.tight_layout()
plt.show()

## 7. Augmented Metrics Table

Drop-in replacement for the standard metrics table that adds Bayesian uncertainty.
`compute_bayesian_metrics()` wraps `compute_metrics()` and appends credible intervals,
P(Sharpe > 0), and Bayes factors.

In [ ]:
bayes_results = []
for r in results:
    if r["equity"].empty:
        continue
    bm = compute_bayesian_metrics(r["equity"])
    bm["label"] = r["label"]
    bayes_results.append(bm)

print(format_bayesian_table(bayes_results))

In [ ]:
# DataFrame view for further analysis
bayes_df = pd.DataFrame(bayes_results).set_index("label")[
    ["sharpe", "sharpe_ci_lower", "sharpe_ci_upper", "p_positive_sharpe",
     "bayes_factor", "bf_interpretation",
     "hit_rate", "hit_rate_ci_lower", "hit_rate_ci_upper", "p_hit_above_50",
     "cagr", "max_dd"]
]
bayes_df

## 8. Regime Prior Updating

Instead of hard tercile-based regime cutoffs (BULL/BEAR/CHOP), we use **Beta updating**
to track P(bull regime) as a smooth, uncertainty-aware probability.

Each day, we observe whether BTC's rolling return is positive ("win") or negative ("loss")
and update a Beta distribution. The resulting probability transitions smoothly and carries
uncertainty — early in the window, the CI is wide; as evidence accumulates, it narrows.

In [ ]:
REGIME_LOOKBACK = 21

btc_rolling_ret = btc["ret"].rolling(REGIME_LOOKBACK).sum()
btc_positive = (btc_rolling_ret > 0).astype(int)

# Online Beta updating with a decaying prior
alpha_prior, beta_prior = 2.0, 2.0  # mildly informative: centred at 50%
decay = 0.98  # exponential decay on pseudo-counts (forgets old data)

p_bull = np.full(len(btc), np.nan)
p_bull_lower = np.full(len(btc), np.nan)
p_bull_upper = np.full(len(btc), np.nan)

a, b = alpha_prior, beta_prior
for i in range(REGIME_LOOKBACK, len(btc)):
    obs = btc_positive.iloc[i]
    if pd.isna(obs):
        continue
    a = decay * a + obs
    b = decay * b + (1 - obs)
    dist = sp_stats.beta(a, b)
    p_bull[i] = dist.mean()
    p_bull_lower[i] = dist.ppf(0.10)
    p_bull_upper[i] = dist.ppf(0.90)

btc["p_bull"] = p_bull
btc["p_bull_lower"] = p_bull_lower
btc["p_bull_upper"] = p_bull_upper

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={"height_ratios": [1, 1.2]})

# Top: BTC price
ax1.plot(btc.index, btc["close"], color=NAVY, lw=1)
ax1.set_ylabel("BTC Price (USD)")
ax1.set_yscale("log")
ax1.set_title("BTC Price & Bayesian Regime Probability", fontweight="bold", fontsize=12)

# Bottom: P(bull) with CI
valid = btc["p_bull"].dropna()
ax2.fill_between(btc.index, btc["p_bull_lower"], btc["p_bull_upper"],
                 alpha=0.2, color=TEAL, label="80% CI")
ax2.plot(btc.index, btc["p_bull"], color=TEAL, lw=1.2, label="P(bull regime)")
ax2.axhline(0.5, color=GRAY, ls=":", alpha=0.5)
ax2.axhline(0.6, color=GOLD, ls="--", alpha=0.4, lw=0.8)
ax2.axhline(0.4, color=RED, ls="--", alpha=0.4, lw=0.8)
ax2.set_ylabel("P(Bull Regime)")
ax2.set_ylim(0, 1)
ax2.legend(loc="upper right", frameon=True, facecolor="white", fontsize=9)

fig.tight_layout()
plt.show()

## 9. Summary

**Key takeaways from Bayesian evaluation:**

1. **Point Sharpe is misleading** — the credible intervals show how much uncertainty
   remains, especially for strategies with < 3 years of data.

2. **P(A beats B) is the honest comparison** — when two strategies have similar Sharpes,
   the posterior probability is often near 50%, meaning we genuinely can't distinguish them.

3. **Bayes factors provide an evidence scale** — not just "significant / not significant"
   but a continuous measure of how strongly the data support positive edge.

4. **Parameter robustness via BMA** — if the max-Sharpe parameter is isolated (effective_n_params ~ 1),
   it's likely overfit. A broad plateau (effective_n_params > 3) is what we want.

5. **Smooth regime probabilities** — the Beta-updating approach gives uncertainty-aware
   regime estimates that degrade gracefully, instead of hard cutoffs that whipsaw.

**Integration path:** Use `compute_bayesian_metrics()` as a drop-in replacement for
`compute_metrics()` in any notebook. Use `p_a_beats_b()` in ablation studies.
Use `parameter_robustness()` whenever doing parameter sweeps.